# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python) 

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [2]:
# importar librerías
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest
from scipy import stats
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns


In [3]:
# cargar archivos
orders = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv')
catalog = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv')
marketing = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv')

In [4]:
# explorar datasets
orders.head()

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28


In [5]:
catalog.head()

,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"


In [6]:
marketing.head()

,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40


---

### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas 

---

In [7]:
print(orders.dtypes)

id_pedido              object
id_usuario             object
fecha_hora_pedido      object
pais                   object
dispositivo            object
fuente_referencia      object
nombre_producto        object
categoria_producto     object
cantidad              float64
precio_unitario       float64
monto_descuento       float64
monto_total           float64
dtype: object


In [8]:
orders['fecha_hora_pedido'] = pd.to_datetime(orders['fecha_hora_pedido'], errors='coerce')

In [9]:
print(marketing.dtypes)

fecha          object
pais           object
id_campaña     object
canal          object
gasto         float64
dtype: object


In [10]:
marketing['fecha'] = pd.to_datetime(marketing['fecha'], errors ='coerce')

In [11]:
print(orders.describe())
print(catalog.describe())
print(marketing.describe())

           cantidad  precio_unitario  monto_descuento   monto_total
count  25050.000000     25050.000000     25050.000000  2.510000e+04
mean       7.092735       259.305549         4.500798  2.072680e+03
std      296.277003       138.726461         5.223010  9.894995e+04
min       -2.000000        20.030000         0.000000 -4.926500e+02
25%        1.000000       138.377500         0.000000  1.805075e+02
50%        2.000000       258.715000         0.000000  3.417500e+02
75%        2.000000       380.332500        10.000000  5.185800e+02
max    20000.000000       499.960000        15.000000  8.840200e+06
       costo_unitario
count        7.000000
mean       102.252857
std        111.011563
min         10.120000
25%         16.905000
50%         25.210000
75%        182.975000
max        280.680000
            gasto
count  1620.00000
mean   1772.74292
std     734.43294
min     501.11000
25%    1128.03000
50%    1782.42500
75%    2420.68500
max    2999.36000


In [12]:
orders = orders[orders['monto_total'] > 0]

orders = orders.reset_index(drop=True)

print(orders.describe())

           cantidad  precio_unitario  monto_descuento   monto_total
count  25046.000000     25046.000000     25046.000000  2.509600e+04
mean       7.094067       259.304400         4.500719  2.073056e+03
std      296.300643       138.715188         5.223232  9.895783e+04
min        1.000000        20.030000         0.000000  5.240000e+00
25%        1.000000       138.405000         0.000000  1.805925e+02
50%        2.000000       258.715000         0.000000  3.418050e+02
75%        2.000000       380.272500        10.000000  5.185950e+02
max    20000.000000       499.960000        15.000000  8.840200e+06


In [13]:
orders['monto_calculado'] = orders['precio_unitario'] * orders['cantidad']

orders['diferencia'] = orders['monto_total'] - orders['monto_calculado']

inconsistencias = orders[orders['diferencia'].abs() > 0.01]

print(f"Total de registros con montos inconsistentes: {len(inconsistencias)}")
print(inconsistencias[['precio_unitario', 'cantidad', 'monto_total', 'monto_calculado', 'diferencia']].head())

Total de registros con montos inconsistentes: 13065
   precio_unitario  cantidad  monto_total  monto_calculado  diferencia
1           176.86       1.0       171.86           176.86       -5.00
2           102.99       2.0       195.99           205.98       -9.99
3           257.87       1.0       242.87           257.87      -15.00
8           477.27       2.0       949.54           954.54       -5.00
9           339.30       1.0       334.30           339.30       -5.00


In [14]:
orders['monto_total'] = orders['precio_unitario'] * orders['cantidad']

orders['monto_total'] = orders['monto_total'].round(2)

print(orders[['precio_unitario', 'cantidad', 'monto_total']].head())

   precio_unitario  cantidad  monto_total
0           332.69       2.0       665.38
1           176.86       1.0       176.86
2           102.99       2.0       205.98
3           257.87       1.0       257.87
4           336.28       1.0       336.28


In [15]:

print("Total de filas duplicadas:", orders.duplicated().sum())

print(orders[orders.duplicated()].head())


Total de filas duplicadas: 100
         id_pedido id_usuario fecha_hora_pedido       pais dispositivo  \
24996  order_22936  user_4028        2025-06-30  Argentina     desktop   
24997  order_13710  user_4466        2025-03-20     Mexico     desktop   
24998  order_14562  user_6590        2025-03-23     Mexico      mobile   
24999  order_11537  user_3115        2025-03-08  Argentina     desktop   
25000   order_4533  user_7944        2025-02-16  Argentina      mobile   

      fuente_referencia       nombre_producto categoria_producto  cantidad  \
24996            social      Vacuum-Pro-Black              Hogar       2.0   
24997            social       Jacket-Winter-M               Moda       2.0   
24998       paid_search  Tablet-Standard-64GB        Electronica       1.0   
24999           organic      Vacuum-Pro-Black              Hogar       1.0   
25000           organic        Blender-XL-Red              Hogar       1.0   

       precio_unitario  monto_descuento  monto_total  m

In [16]:
orders = orders.drop_duplicates()

orders = orders.reset_index(drop=True)

In [17]:



print(orders['dispositivo'].value_counts(dropna=False))
cols_categoricas = [
    "id_pedido", 
    "id_usuario", 
    "pais", 
    "dispositivo", 
    "fuente_referencia", 
    "nombre_producto", 
    "categoria_producto"
]
for col in cols_categoricas:
    print(f"\n--- Valores únicos en '{col}' ---")
    print(orders[col].unique())






desktop    12707
mobile     12269
NaN           20
Name: dispositivo, dtype: int64

--- Valores únicos en 'id_pedido' ---
['order_0' 'order_1' 'order_2' ... 'order_24997' 'order_24998'
 'order_24999']

--- Valores únicos en 'id_usuario' ---
['user_6993' 'user_1329' 'user_3194' ... 'user_7965' 'user_76' 'user_670']

--- Valores únicos en 'pais' ---
['Argentina' 'Mexico' 'Colombia' 'mexico' 'colombia' 'argentina' nan]

--- Valores únicos en 'dispositivo' ---
['desktop' 'mobile' nan]

--- Valores únicos en 'fuente_referencia' ---
['organic' 'paid_search' 'social' nan]

--- Valores únicos en 'nombre_producto' ---
['Jacket-Winter-M' 'Tablet-Standard-64GB' 'Blender-XL-Red'
 'Laptop-Gaming-16GB' 'Sneakers-Urban-42' 'Phone-Pro-128GB'
 'Vacuum-Pro-Black' nan]

--- Valores únicos en 'categoria_producto' ---
['Moda' 'Electronica' 'Hogar' nan]


In [18]:
orders['pais'] = orders['pais'].str.strip()
orders['pais'] = orders['pais'].str.title()  

---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [19]:
# exportar datasets
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)? 
- ¿Cuál es el costo total? 
- ¿Cuánto se ha invertido en marketing? 
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden? 
- ¿Cuál es la cantidad promedio de productos por orden? 
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal? 

In [52]:
ingreso_total = orders['monto_total'].sum()
orders_catalog = orders.merge(catalog[['nombre_producto','costo_unitario']], on='nombre_producto', how='left')
orders_catalog['costo_total'] = catalog['costo_unitario'] * orders_catalog['cantidad']
costo_producto_total = orders_catalog['costo_total'].sum()
inversion_total = marketing['gasto'].sum()
costo_total = costo_producto_total + inversion_total
ganancia = ingreso_total - costo_total
porcentaje_margen = (ganancia/ingreso_total)*100
print(f"--- Resumen de rentabilidad del negocio ---")
print(f"Ingreso Total (Revenue):          ${ingreso_total:,.2f}")
print(f"Costo Producto Total:              ${costo_producto_total:,.2f}")
print(f"Inversion Total:                   ${inversion_total:,.2f}")
print(f"Ganancia:                          ${ganancia:,.2f}")
print(f"Porcentaje margen:                 %{porcentaje_margen:,.2f}")

if ganancia > 0:
    print("\n Conclusion: El negocio es rentable")
else:
    print("\n Conclusion: El negocio no es rentable")

--- Resumen de rentabilidad del negocio ---
Ingreso Total (Revenue):          $52,079,302.37
Costo Producto Total:              $1,210.97
Inversion Total:                   $2,871,843.53
Ganancia:                          $49,206,247.87
Porcentaje margen:                 %94.48

 Conclusion: El negocio es rentable


In [53]:
ticket_promedio = orders['monto_total'].mean()
promedio_productos = orders['cantidad'].mean()
producto_mas_vendido_unidades = orders.groupby('nombre_producto')['cantidad'].sum().idxmax()
unidades_totales = orders.groupby('nombre_producto')['cantidad'].sum().max()
gasto_por_canal = marketing.groupby('canal')['gasto'].sum().sort_values(ascending=False)

print(f"Resumen de los del comportamiento de ventas")
print(f"Ticket Promedio               ${ticket_promedio:,.2f}")
print(f"Producto mas vendido          {producto_mas_vendido_unidades}")
print(f"Unidades Totales              {unidades_totales:,.2f}")
print(f"Gasto por canal               {gasto_por_canal}")

Resumen de los del comportamiento de ventas
Ticket Promedio               $2,087.68
Producto mas vendido          Laptop-Gaming-16GB
Unidades Totales              144,198.00
Gasto por canal               canal
social         918043.21
organic        913533.01
paid_search    863088.21
Name: gasto, dtype: float64


---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [54]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [55]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()


,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [56]:

# PARTE 1: Totales del funnel
# ======================
query_totals = '''
SELECT 
    nombre_evento,
    COUNT(DISTINCT id_usuario) AS usuarios_unicos
FROM
    events
GROUP BY 
    nombre_evento
ORDER BY 
    CASE nombre_evento
        WHEN 'first_visit' THEN 1
        WHEN 'select_item' THEN 2
        WHEN 'add_to_cart' THEN 3
        WHEN 'begin_checkout' THEN 4
        WHEN 'add_payment_info' THEN  5
        WHEN 'purchase' THEN 6
        ELSE 7
    END;
'''

totals = pd.read_sql(query_totals, con=engine)
totals


,nombre_evento,usuarios_unicos
0,first_visit,7796
1,select_item,7582
2,add_to_cart,7634
3,begin_checkout,7208
4,add_payment_info,6250
5,purchase,6240


In [75]:

# PARTE 2: Análisis de conversión del Funnel
# ==========================================

query_conversion = '''
WITH funnel_totales AS (
    SELECT
        nombre_evento,
        COUNT(DISTINCT id_usuario) AS usuarios_unicos,
        CASE nombre_evento
            WHEN 'first_visit'      THEN 1
            WHEN 'select_item'      THEN 2
            WHEN 'add_to_cart'      THEN 3
            WHEN 'begin_checkout'   THEN 4  
            WHEN 'add_payment_info' THEN 5  
            WHEN 'purchase'         THEN 6
            ELSE 7
        END AS orden
    FROM
        events
    GROUP BY 
        nombre_evento
)
SELECT
    nombre_evento,
    usuarios_unicos,
    LAG(usuarios_unicos) OVER (ORDER BY orden) AS usuarios_paso_anterior,
     
    ROUND(
        (usuarios_unicos::DECIMAL / NULLIF(LAG(usuarios_unicos) OVER (ORDER BY orden), 0)) * 100, 2
    ) AS pct_conversion_paso_a_paso,
    
    LAG(usuarios_unicos) OVER (ORDER BY orden) - usuarios_unicos AS usuarios_perdidos,
    
    ROUND(
        ((LAG(usuarios_unicos) OVER (ORDER BY orden) - usuarios_unicos)::DECIMAL / 
         NULLIF(LAG(usuarios_unicos) OVER (ORDER BY orden), 0)) * 100, 2
    ) AS pct_drop_off,
    
    ROUND(
        (usuarios_unicos::DECIMAL / FIRST_VALUE(usuarios_unicos) OVER (ORDER BY orden)) * 100, 2
    ) AS pct_conversion_total
FROM
    funnel_totales
ORDER BY
    orden;
'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion

,nombre_evento,usuarios_unicos,usuarios_paso_anterior,pct_conversion_paso_a_paso,usuarios_perdidos,pct_drop_off,pct_conversion_total
0,first_visit,7796,NaN,NaN,NaN,NaN,100.00
1,select_item,7582,7796.0,97.26,214.0,2.74,97.26
2,add_to_cart,7634,7582.0,100.69,-52.0,-0.69,97.92
3,begin_checkout,7208,7634.0,94.42,426.0,5.58,92.46
4,add_payment_info,6250,7208.0,86.71,958.0,13.29,80.17
5,purchase,6240,6250.0,99.84,10.0,0.16,80.04


---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users` 
- `user_activity` 

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [58]:
# Explorar tabla users
# =========================

query_users = '''

SELECT *
FROM users;

'''


users = pd.read_sql(query_users, con=engine)
users.head(3)


,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [59]:
# Explorar tabla users con fecha convertida a DATE
# =================================================
query_users = '''
SELECT 
    id_usuario,
    país,
    dispositivo,
    tipo_plan,
    CAST(fecha_registro AS DATE) AS fecha_registro
FROM 
    users;
'''

users = pd.read_sql(query_users, con=engine)
users.head(3)

,id_usuario,país,dispositivo,tipo_plan,fecha_registro
0,user_0,Mexico,mobile,free,2025-01-29
1,user_1,Mexico,mobile,free,2025-01-07
2,user_2,Argentina,mobile,free,2025-03-12


In [60]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT *
FROM user_activity;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1


In [61]:
# Retención por cohortes
# ======================

query_cohort_retention_final = '''
WITH cohortes_usuarios AS(
    SELECT
        id_usuario,
        DATE_TRUNC('month', CAST(fecha_registro AS DATE)) AS mes_cohorte
    FROM
        users
 )
 SELECT
     TO_CHAR(c.mes_cohorte, 'YYYY-MM') AS cohorte,
     a.dias_despues_registro,
     COUNT(DISTINCT c.id_usuario) AS total_usuarios_cohorte,
     COUNT(DISTINCT CASE WHEN a.activo = 1 THEN a.id_usuario END) AS usuarios_activos,
     ROUND(
         (COUNT(DISTINCT CASE WHEN a.activo = 1 THEN a.id_usuario END)::DECIMAL / NULLIF(COUNT(DISTINCT c.id_usuario), 0)) * 100, 2
     ) AS pct_retention
    FROM 
        cohortes_usuarios c
    INNER JOIN
        user_activity a ON c.id_usuario = a.id_usuario
    GROUP BY
        c.mes_cohorte,
        a.dias_despues_registro
    ORDER BY
        c.mes_cohorte,
        a.dias_despues_registro;
'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

,cohorte,dias_despues_registro,total_usuarios_cohorte,usuarios_activos,pct_retention
0,2025-01,7,1627,697,42.84
1,2025-01,14,1627,668,41.06
2,2025-01,21,1627,656,40.32
3,2025-01,28,1627,671,41.24
4,2025-02,7,1444,611,42.31
5,2025-02,14,1444,609,42.17
6,2025-02,21,1444,635,43.98
7,2025-02,28,1444,575,39.82
8,2025-03,7,1636,677,41.38
9,2025-03,14,1636,705,43.09


In [62]:
# Retención semanal por cohortes (W1, W2, W3)
# ============================================

query_retencion_semanal = '''
WITH base_cohortes AS (
    -- 1. Determinar el mes de registro de cada usuario (Cohorte)
    SELECT 
        id_usuario,
        DATE_TRUNC('month', CAST(fecha_registro AS DATE)) AS mes_cohorte
    FROM 
        users
),
actividad_semanal AS (
    -- 2. Vincular con la actividad agregando flags de retención por semana
    SELECT 
        c.mes_cohorte,
        c.id_usuario,
        MAX(CASE WHEN a.dias_despues_registro = 7  AND a.activo = 1 THEN 1 ELSE 0 END) AS retenido_w1,
        MAX(CASE WHEN a.dias_despues_registro = 14 AND a.activo = 1 THEN 1 ELSE 0 END) AS retenido_w2,
        MAX(CASE WHEN a.dias_despues_registro = 21 AND a.activo = 1 THEN 1 ELSE 0 END) AS retenido_w3
    FROM 
        base_cohortes c
    LEFT JOIN 
        user_activity a ON c.id_usuario = a.id_usuario
    GROUP BY 
        c.mes_cohorte,
        c.id_usuario
)
-- 3. Calcular totales y porcentajes por cohorte
SELECT 
    TO_CHAR(mes_cohorte, 'YYYY-MM') AS cohorte,
    COUNT(DISTINCT id_usuario) AS usuarios_iniciales,
    
    -- Conteo de usuarios retenidos por semana
    SUM(retenido_w1) AS retenido_w1,
    SUM(retenido_w2) AS retenido_w2,
    SUM(retenido_w3) AS retenido_w3,
    
    -- Porcentajes de retención por semana
    ROUND((SUM(retenido_w1)::DECIMAL / COUNT(DISTINCT id_usuario)) * 100, 2) AS semana_1,
    ROUND((SUM(retenido_w2)::DECIMAL / COUNT(DISTINCT id_usuario)) * 100, 2) AS semana_2,
    ROUND((SUM(retenido_w3)::DECIMAL / COUNT(DISTINCT id_usuario)) * 100, 2) AS semana_3
FROM 
    actividad_semanal
GROUP BY 
    mes_cohorte
ORDER BY 
    mes_cohorte;
'''

retencion_semanal = pd.read_sql(query_retencion_semanal, con=engine)
retencion_semanal

,cohorte,usuarios_iniciales,retenido_w1,retenido_w2,retenido_w3,semana_1,semana_2,semana_3
0,2025-01,1627,697,668,656,42.84,41.06,40.32
1,2025-02,1444,611,609,635,42.31,42.17,43.98
2,2025-03,1636,677,705,690,41.38,43.09,42.18
3,2025-04,1606,680,697,663,42.34,43.40,41.28
4,2025-05,1687,695,676,706,41.20,40.07,41.85


---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado** 
4. **Interpretar el resultado**  

---
Hipótesis estadística
   - **H₀ (Hipótesis nula):** ...
   - **H₁ (Hipótesis alternativa):** ...
   
**Test estadístico:** ...  
**Nivel de significancia alpha:** ...

In [74]:
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest

# 1. Cargar el dataset
url = 'https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv'
df_exp = pd.read_csv(url)

# Limpiar posibles espacios en blanco en la columna 'variante'
df_exp['variante'] = df_exp['variante'].astype(str).str.strip()

# 2. Resumen descriptivo
resumen = df_exp.groupby('variante')['convirtio'].agg(
    total_usuarios='count',
    conversiones='sum',
    tasa_conversion='mean'
).reset_index()

# 3. Identificación dinámica de los grupos
grupo_control = resumen[resumen['variante'].str.lower() == 'control']

# El grupo tratamiento será cualquier fila que no sea 'control'
grupo_tratamiento = resumen[resumen['variante'].str.lower() != 'control']

# Extraer valores métricos de forma segura
n_control = grupo_control['total_usuarios'].values[0]
conv_control = grupo_control['conversiones'].values[0]
p_control = grupo_control['tasa_conversion'].values[0]

n_treatment = grupo_tratamiento['total_usuarios'].values[0]
conv_treatment = grupo_tratamiento['conversiones'].values[0]
p_treatment = grupo_tratamiento['tasa_conversion'].values[0]

nombre_tratamiento = grupo_tratamiento['variante'].values[0]

# Calculate Lift relativo (%)
lift_relativo = ((p_treatment - p_control) / p_control) * 100

print("--- Resumen de Conversión por Variante ---")
print(resumen.to_string(index=False))
print(f"\nVariante de prueba detectada: '{nombre_tratamiento}'")
print(f"Lift relativo ({nombre_tratamiento} vs Control): {lift_relativo:.2f}%")

# 4. Test Z de Proporciones (Tratamiento vs Control)
z_stat, p_value = proportions_ztest(
    count=[conv_treatment, conv_control], 
    nobs=[n_treatment, n_control]
)

print("\n--- Resultados del Test Estadístico ---")
print(f"Estadístico Z: {z_stat:.4f}")
print(f"P-valor:       {p_value:.4e}")

# 5. Regla de decisión
alpha = 0.05
if p_value < alpha:
    print(f"\nDecisión: Rechazar H0 (p-valor < {alpha}). La diferencia en la tasa de conversión es estadísticamente significativa.")
else:
    print(f"\nDecisión: No rechazar H0 (p-valor >= {alpha}). No existe diferencia estadísticamente significativa entre los grupos.")

--- Resumen de Conversión por Variante ---
   variante  total_usuarios  conversiones  tasa_conversion
    control            4965           779         0.156898
tratamiento            5035           820         0.162860

Variante de prueba detectada: 'tratamiento'
Lift relativo (tratamiento vs Control): 3.80%

--- Resultados del Test Estadístico ---
Estadístico Z: 0.8133
P-valor:       4.1606e-01

Decisión: No rechazar H0 (p-valor >= 0.05). No existe diferencia estadísticamente significativa entre los grupos.


---

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión. 

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc

---

## 🚀 Entrega Final

Comparte el acceso a tu Dashboard para revisión.   
Puedes entregar el Dashboard utilizando **Power BI o Tableau**.

Incluye **uno de los siguientes**:

- 🔗 Link público del dashboard publicado en **Power BI Service o Tableau Public / Tableau Cloud**
- 🔗 Link de **Google Drive o OneDrive** con el archivo del proyecto (`.pbix`) y los 3 csvs limpios.


### 📎 Enlace del Dashboard

In [ ]:
# (Pega aquí tu link)
# link de power bi o tableau
# https://drive.google.com/drive/folders/1ZEznFSQqX0PQcxwnS-PPGKcrE5zpN2Mn?usp=sharing